# Train AI Quoridor bằng RL Actor-Critic (A2C)

Notebook này huấn luyện một agent Quoridor với thuật toán **Advantage Actor-Critic (A2C)**, tái sử dụng engine và mạng của project:

- `quoridor_ai.core.engine` — luật chơi, action 0..208 (0-80: đi pawn, 81-144: tường ngang, 145-208: tường dọc)
- `quoridor_ai.core.encoding.encode_v3` — 16 planes đầu vào
- `quoridor_ai.model.PolicyValueNet` — ResNet có sẵn 2 head: **actor** (policy 209 actions) và **critic** (value V(s))

Pipeline: **self-play thu trajectory → tính advantage GAE → cập nhật actor (policy loss) + critic (value loss) + entropy bonus**.

In [32]:
import copy, csv, json, math, os, random, time
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from quoridor_ai.core.engine import State, ACTION_SIZE, legal_actions, apply_unchecked, dist_to_goal
from quoridor_ai.core.encoding import canonical_actions, decanonical_action, encode_v3
from quoridor_ai.model import PolicyValueNet, masked_policy
from quoridor_ai.runtime import resolve_device

device = resolve_device()
torch.manual_seed(42); np.random.seed(42); random.seed(42)
RUN_ID = os.environ.get('A2C_RUN_ID') or time.strftime('a2c_%Y%m%d_%H%M%S')
RUN_DIR = Path('results') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('results:', RUN_DIR)
print('device:', device)

results: results\a2c_20261107_115542
device: cuda


## 1. Siêu tham số (hyperparameters)

Mỗi tham số sẽ được giải thích chi tiết ở cuối notebook.

In [33]:
HP = dict(
    channels        = 64,     # kênh trong ResNet
    blocks          = 6,      # số residual block
    iterations      = 1024+512,    # số vòng self-play + học
    games_per_iter  = 64,     # số ván tự chơi mỗi vòng
    max_plies       = 256,    # giới hạn nước đi mỗi ván (tránh ván dài vô hạn)
    repetition_draw = True,  # repetition là hoà trong rollout, không đổi luật engine
    avoid_repetition = True, # ưu tiên action không quay về state đã gặp nếu còn lựa chọn
    reward_shaping = True,   # potential shaping theo shortest path BFS
    potential_scale = 0.1,   # biên độ shaping nhỏ so với reward thắng/thua
    wall_value_weight = 0.25, # giữ lại tường trừ khi chúng cải thiện thế đường đi
    # --- rollout thu dữ liệu ---
    ep_temp         = 1.0,    # softmax temperature khi chọn action lúc khám phá
    temp_floor_move = 12,     # sau 12 nước đầu giảm temp xuống 0.5 (bớt ngẫu nhiên)
    dirichlet_alpha = 0.5,    # noise Dirichlet trộn vào policy gốc (khám phá)
    noise_frac      = 0.5,   # tỉ lệ trọng số của noise Dirichlet
    noise_plies     = 8,      # dành cho thử nghiệm; self-play A2C tắt Dirichlet noise
    pawn_action_prior = 0.75,  # prior đi quân khi logits ban đầu ngang nhau
    # --- A2C ---
    gamma           = 0.999,   # hệ số chiết khấu tương lai
    gae_lambda      = 0.99,   # lambda của GAE (Generalized Advantage Estimation)
    policy_coef     = 1.0,    # hệ số policy loss
    value_coef      = 0.5,    # hệ số value loss
    entropy_coef    = 0.0025,   # hệ số entropy bonus (khuyến khích khám phá)
    # --- tối ưu ---
    lr              = 1e-4,   # learning rate AdamW
    epochs_per_iter = 1,      # on-policy A2C dùng mỗi trajectory một lần
    batch_size      = 1024,  # gom rollout thành ít minibatch, giữ A2C gần on-policy
    teacher_batch_size = 32,
    grad_clip       = 0.5,    # clip gradient toàn cục
    weight_decay    = 1e-4,
    opponent_pool_size = 8,
    opponent_snapshot_interval = 5,
    heuristic_opponent_fraction = 0.85, # tỉ lệ ván gặp heuristic
    checkpoint_opponent_fraction = 0, # phần còn lại gặp checkpoint cũ/self-play
    heuristic_wall_cost = 0.5, # chỉ đặt tường khi làm chậm đối thủ đáng kể
    teacher_pretrain = True, # học bắt chước teacher trước khi bắt đầu A2C
    teacher_games = 32, # số ván teacher-vs-teacher để tạo ví dụ
    teacher_epochs = 8, # số epoch supervised trên các ví dụ teacher
    teacher_wall_candidates = 128, # chấm toàn bộ nước tường hợp lệ
    teacher_search_width = 4, # giữ top action để tìm phản đòn một ply
    evaluation_games = 100,
    resume_from = r'results/a2c_20261107_064618/a2c_quoridor.pt',  # run mới, tách biệt checkpoint kiến trúc cũ
)
with (RUN_DIR / 'config.json').open('w', encoding='utf-8') as stream:
    json.dump(HP, stream, ensure_ascii=False, indent=2)
display(HP)

{'channels': 64,
 'blocks': 6,
 'iterations': 1536,
 'games_per_iter': 64,
 'max_plies': 256,
 'repetition_draw': True,
 'avoid_repetition': True,
 'reward_shaping': True,
 'potential_scale': 0.1,
 'wall_value_weight': 0.25,
 'ep_temp': 1.0,
 'temp_floor_move': 12,
 'dirichlet_alpha': 0.5,
 'noise_frac': 0.5,
 'noise_plies': 8,
 'pawn_action_prior': 0.75,
 'gamma': 0.999,
 'gae_lambda': 0.99,
 'policy_coef': 1.0,
 'value_coef': 0.5,
 'entropy_coef': 0.0025,
 'lr': 0.0001,
 'epochs_per_iter': 1,
 'batch_size': 1024,
 'teacher_batch_size': 32,
 'grad_clip': 0.5,
 'weight_decay': 0.0001,
 'opponent_pool_size': 8,
 'opponent_snapshot_interval': 5,
 'heuristic_opponent_fraction': 0.85,
 'checkpoint_opponent_fraction': 0,
 'heuristic_wall_cost': 0.5,
 'teacher_pretrain': True,
 'teacher_games': 32,
 'teacher_epochs': 8,
 'teacher_wall_candidates': 128,
 'teacher_search_width': 4,
 'evaluation_games': 100,
 'resume_from': 'results/a2c_20261107_064618/a2c_quoridor.pt'}

## 2. Khởi tạo mạng

`PolicyValueNet` xuất ra:
- `logits` có shape `(B, 209)` — **actor**
- `value` có shape `(B, 1)` — **critic**, dự đoán giá trị thế cờ trong `[-1, 1]` (+1 = người hiện tại thắng)

In [34]:
net = PolicyValueNet(HP['channels'], HP['blocks'], planes=16).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
history = []
start_iteration = 0
teacher_pretrained_for = None
if HP['resume_from']:
    saved = torch.load(HP['resume_from'], map_location=device, weights_only=True)
    net.load_state_dict(saved['model'])
    if 'optimizer' in saved:
        opt.load_state_dict(saved['optimizer'])
    history = list(saved.get('history', []))
    start_iteration = int(saved.get('iteration', len(history)))
print('tham số trainable:', sum(p.numel() for p in net.parameters()))

tham số trainable: 631398


## 3. Self-play thu trajectory

Mỗi nước đi, ta lưu lại observation, legal mask, action canonical đã thực sự chọn, người đi, cờ action có thuộc policy đang train và reward theo góc nhìn người đi.

Với encoder v3, bàn cờ và action/mask đều phải dùng cùng canonical frame. Self-play cùng mạng học action của cả hai bên; khi dùng checkpoint cũ, chỉ actor của mạng hiện tại nhận policy gradient.

In [35]:
@dataclass
class A2CStep:
    observation: np.ndarray
    legal_mask: np.ndarray
    action: int
    player: int
    train_actor: bool
    reward: float
    temperature: float

def state_key(s):
    # ply được xử lý riêng bởi max_plies; lặp cùng vị trí vẫn là repetition.
    return (s.p0, s.p1, s.walls0, s.walls1, s.h, s.v, s.player)

def repetition_safe_actions(s, seen=None):
    actions = legal_actions(s)
    if seen is None:
        return actions
    fresh = [a for a in actions if a >= 81 or state_key(apply_unchecked(s, a)) not in seen]
    return fresh or actions

def legal_mask(s, seen=None):
    # encode_v3 lật bàn cho player 1; mask/action phải dùng cùng canonical frame.
    actions = canonical_actions(repetition_safe_actions(s, seen), s.player)
    mask = np.zeros(ACTION_SIZE, dtype=np.bool_)
    mask[actions] = True
    return mask

def action_prior_logits(logits, masks, hp=None):
    hp = HP if hp is None else hp
    pawn_prior = float(hp['pawn_action_prior'])
    if not np.isfinite(pawn_prior) or not 0 < pawn_prior < 1:
        raise ValueError('pawn_action_prior phải hữu hạn và nằm giữa 0 và 1')
    masks = masks.bool()
    action_ids = torch.arange(ACTION_SIZE, device=logits.device)
    pawn_count = (masks & (action_ids < 81)).sum(dim=-1, keepdim=True).clamp_min(1)
    wall_count = (masks & (action_ids >= 81)).sum(dim=-1, keepdim=True).clamp_min(1)
    pawn_offset = math.log(pawn_prior) - pawn_count.to(logits.dtype).log()
    wall_offset = math.log1p(-pawn_prior) - wall_count.to(logits.dtype).log()
    offsets = torch.where(action_ids < 81, pawn_offset, wall_offset)
    return logits + offsets

@torch.no_grad()
def select_action(net, s, temperature=1.0, sample=True, noise=False, hp=None, seen=None):
    hp = HP if hp is None else hp
    mask = legal_mask(s, seen if hp.get('avoid_repetition', True) else None)
    legal = np.flatnonzero(mask)
    if legal.size == 0:
        raise ValueError('Không có action hợp lệ ở state chưa kết thúc')
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature phải hữu hạn và > 0')
    net.eval()
    x = torch.from_numpy(encode_v3(s)).unsqueeze(0).to(device)
    logits, value = net(x)
    if not torch.isfinite(logits).all() or not torch.isfinite(value).all():
        raise FloatingPointError('Policy logits/value có NaN hoặc Inf')
    mask_tensor = torch.as_tensor(mask, device=device).unsqueeze(0)
    adjusted_logits = action_prior_logits(logits.float() / temperature, mask_tensor, hp)
    legal_logits = adjusted_logits[0, legal]
    probs_legal = torch.softmax(legal_logits, dim=0).cpu().numpy().astype(np.float64)
    if not np.isfinite(probs_legal).all() or probs_legal.sum() <= 0:
        raise FloatingPointError('Policy probability không thể chuẩn hoá')
    probs_legal /= probs_legal.sum()
    if noise:
        alpha = float(hp['dirichlet_alpha'])
        fraction = float(hp['noise_frac'])
        if alpha <= 0 or not 0 <= fraction <= 1:
            raise ValueError('Dirichlet parameters không hợp lệ')
        for group in (legal < 81, legal >= 81):
            if group.any():
                group_mass = probs_legal[group].sum()
                dirichlet = np.random.dirichlet(np.full(int(group.sum()), alpha))
                probs_legal[group] = ((1 - fraction) * probs_legal[group]
                                      + fraction * group_mass * dirichlet)
        probs_legal /= probs_legal.sum()
    probs = np.zeros(ACTION_SIZE, dtype=np.float64)
    probs[legal] = probs_legal
    canonical_action = int(np.random.choice(legal, p=probs_legal)) if sample else int(legal[np.argmax(probs_legal)])
    real_action = decanonical_action(canonical_action, s.player)
    if real_action not in legal_actions(s):
        raise RuntimeError('Action sau khi đổi canonical frame là bất hợp lệ')
    return real_action, canonical_action, mask, probs, float(value.item())

def heuristic_state_value(s, perspective, hp):
    if s.winner is not None:
        return 1000.0 if s.winner == perspective else -1000.0
    mine = dist_to_goal(s, perspective)
    theirs = dist_to_goal(s, 1 - perspective)
    mine_walls = s.walls0 if perspective == 0 else s.walls1
    their_walls = s.walls1 if perspective == 0 else s.walls0
    return float((theirs - mine) + 0.25 * (mine_walls - their_walls))

def search_heuristic_action(s, seen=None, hp=None):
    hp = HP if hp is None else hp
    actions = repetition_safe_actions(s, seen if hp.get('avoid_repetition', True) else None)
    if not actions:
        raise ValueError('Teacher search không có action hợp lệ')
    player = s.player
    my_distance = dist_to_goal(s, player)
    opponent_distance = dist_to_goal(s, 1 - player)
    my_walls = s.walls0 if player == 0 else s.walls1
    scored = []
    wall_actions = [action for action in actions if action >= 81]
    wall_limit = int(hp['teacher_wall_candidates'])
    if wall_limit < 0:
        raise ValueError('teacher_wall_candidates không được âm')
    if len(wall_actions) > wall_limit:
        wall_actions = random.sample(wall_actions, wall_limit)
    for action in [a for a in actions if a < 81] + wall_actions:
        next_s = apply_unchecked(s, action)
        if next_s.winner == player:
            return action
        next_my_distance = dist_to_goal(next_s, player)
        next_opponent_distance = dist_to_goal(next_s, 1 - player)
        relative_gain = ((opponent_distance - next_opponent_distance)
                         - (next_my_distance - my_distance))
        if action >= 81:
            urgency = 2.0 if opponent_distance <= 2 else 1.0
            wall_cost = hp['heuristic_wall_cost'] + (10 - my_walls) * 0.1
            relative_gain = urgency * relative_gain - wall_cost
        scored.append((relative_gain, action))
    width = int(hp['teacher_search_width'])
    if width <= 0:
        raise ValueError('teacher_search_width phải lớn hơn 0')
    candidates = sorted(scored, reverse=True)[:width]
    searched = []
    for prior_score, action in candidates:
        next_s = apply_unchecked(s, action)
        seen_after = set() if seen is None else set(seen)
        seen_after.add(state_key(next_s))
        replies = repetition_safe_actions(next_s, seen_after)
        if not replies:
            searched.append((prior_score, action))
            continue
        worst_reply = min(
            heuristic_state_value(apply_unchecked(next_s, reply), player, hp)
            for reply in replies
        )
        searched.append((worst_reply, action))
    best_score = max(score for score, _ in searched)
    return random.choice([action for score, action in searched if score == best_score])

def collect_teacher_examples(hp, games=None):
    games = int(hp['teacher_games'] if games is None else games)
    if games <= 0:
        raise ValueError('teacher_games phải lớn hơn 0')
    examples = []
    for _ in range(games):
        s = State(player=random.randrange(2))
        seen = {state_key(s)}
        while s.winner is None and s.ply < hp['max_plies']:
            mask = legal_mask(s, seen if hp.get('avoid_repetition', True) else None)
            action = search_heuristic_action(s, seen=seen, hp=hp)
            canonical_action = canonical_actions([action], s.player)[0]
            if not mask[canonical_action]:
                raise RuntimeError('Teacher chọn action không nằm trong legal mask')
            examples.append((encode_v3(s), mask, canonical_action))
            s = apply_unchecked(s, action)
            if s.winner is not None:
                break
            key = state_key(s)
            if hp['repetition_draw'] and key in seen:
                break
            seen.add(key)
    if not examples:
        raise ValueError('Teacher không tạo được ví dụ huấn luyện')
    return examples

def behavior_clone(net, opt, hp, examples=None):
    if int(hp['teacher_epochs']) <= 0:
        raise ValueError('teacher_epochs phải lớn hơn 0')
    examples = collect_teacher_examples(hp) if examples is None else examples
    observations = torch.from_numpy(np.stack([x[0] for x in examples]).astype(np.float32))
    masks = torch.from_numpy(np.stack([x[1] for x in examples]).astype(np.bool_))
    actions = torch.tensor([x[2] for x in examples], dtype=torch.long)
    if not masks.gather(1, actions[:, None]).all():
        raise ValueError('Teacher dataset có action ngoài legal mask')
    dataset = TensorDataset(observations, masks, actions)
    loader = DataLoader(dataset, batch_size=hp['teacher_batch_size'], shuffle=True, drop_last=False)
    net.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for epoch in range(int(hp['teacher_epochs'])):
        for obs, legal, target in loader:
            obs, legal, target = obs.to(device), legal.to(device), target.to(device)
            logits, _ = net(obs)
            if not torch.isfinite(logits).all():
                raise FloatingPointError('Teacher policy logits có NaN/Inf')
            adjusted = action_prior_logits(logits, legal, hp)
            logp = F.log_softmax(masked_policy(adjusted, legal), dim=-1)
            loss = F.nll_loss(logp, target)
            if not torch.isfinite(loss):
                raise FloatingPointError('Behavior cloning loss có NaN/Inf')
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), hp['grad_clip'])
            opt.step()
            total_loss += float(loss.detach()) * len(target)
            total_correct += int((logp.argmax(dim=-1) == target).sum())
            total_seen += len(target)
    recalibrate_batch_norm(net, observations, hp)
    metrics = {'teacher_examples': len(dataset), 'teacher_loss': total_loss / max(total_seen, 1),
               'teacher_action_accuracy': total_correct / max(total_seen, 1)}
    print('Behavior cloning:', metrics)
    return metrics

@torch.no_grad()
def recalibrate_batch_norm(net, observations, hp):
    layers = [module for module in net.modules()
              if isinstance(module, torch.nn.BatchNorm2d)]
    if not layers:
        net.eval()
        return
    momenta = [layer.momentum for layer in layers]
    for layer in layers:
        layer.reset_running_stats()
        layer.momentum = None
    net.train()
    try:
        loader = DataLoader(TensorDataset(observations.detach()),
                            batch_size=max(int(hp['batch_size']), 32), shuffle=False)
        for (obs,) in loader:
            net(obs.to(device))
    finally:
        for layer, momentum in zip(layers, momenta):
            layer.momentum = momentum
        net.eval()
    print(f'BatchNorm recalibrated on {len(observations)} states.')

def potential(s, hp=None):
    hp = HP if hp is None else hp
    if s.winner is not None:
        return 0.0
    mine = dist_to_goal(s, s.player)
    theirs = dist_to_goal(s, 1 - s.player)
    mine_walls = s.walls0 if s.player == 0 else s.walls1
    their_walls = s.walls1 if s.player == 0 else s.walls0
    race = (theirs - mine) / 18.0
    wall_reserve = hp.get('wall_value_weight', 0.5) * (mine_walls - their_walls) / 10.0
    return float(np.clip(race + wall_reserve, -1.0, 1.0) * hp['potential_scale'])

def shaping_reward(s, next_s, hp=None, terminal=False):
    hp = HP if hp is None else hp
    if not hp['reward_shaping']:
        return 0.0
    # Phi(next) đang nhìn từ phía đối thủ, nên đảo dấu về phía người vừa đi.
    next_phi = 0.0 if terminal or next_s.winner is not None else -potential(next_s, hp)
    return float(hp['gamma'] * next_phi - potential(s, hp))

class OpponentPool:
    def __init__(self, capacity):
        self.capacity = int(capacity)
        self.snapshots = []
    def __len__(self):
        return len(self.snapshots)
    def add(self, net, iteration, path=None):
        state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        self.snapshots.append((int(iteration), state))
        self.snapshots = self.snapshots[-self.capacity:]
        if path is not None:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            temporary = path + '.tmp'
            torch.save({'model': state, 'iteration': int(iteration), 'hp': dict(HP)}, temporary)
            os.replace(temporary, path)
    def sample(self):
        if not self.snapshots:
            raise ValueError('Opponent pool đang rỗng')
        iteration, state = random.choice(self.snapshots)
        return iteration, {k: v.clone() for k, v in state.items()}
    def load_directory(self, directory, template):
        count = 0
        skipped = 0
        template_state = template.state_dict()
        for path in sorted(Path(directory).glob('a2c_iter_*.pt')):
            data = torch.load(path, map_location='cpu', weights_only=True)
            state = data['model']
            mismatched = [key for key, tensor in template_state.items()
                          if key not in state or tensor.shape != state[key].shape]
            unexpected = [key for key in state if key not in template_state]
            if mismatched or unexpected:
                skipped += 1
                print(f'Bỏ qua checkpoint đối thủ không cùng kiến trúc: {path.name} '
                      f'(shape/key không khớp: {len(mismatched) + len(unexpected)}).')
                continue
            candidate = copy.deepcopy(template)
            candidate.load_state_dict(state)
            self.add(candidate, data.get('iteration', 0))
            count += 1
        if skipped:
            print(f'Opponent pool: đã nạp {count}, bỏ qua {skipped} checkpoint khác kiến trúc.')
        return count

## 4. GAE (Generalized Advantage Estimation)

Mỗi trajectory gồm toàn bộ ply, kể cả nước của đối thủ cũ. Reward thắng/thua nằm ở cuối; potential-based shaping BFS có thể bật/tắt trong `HP`:

- $r_T = z$ (chỉ ở bước cuối), cộng shaping nhỏ theo chênh lệch shortest path
- $\delta_t = r_t - \gamma V(s_{t+1}) - V(s_t)$ vì state kế tiếp nhìn từ phía đối thủ
- GAE đổi dấu qua mỗi lượt: $A_t = \delta_t - \gamma\lambda A_{t+1}$
- Return target cho critic: $G_t = A_t + V(s_t)$

Vì board là zero-sum 2 người đổi lượt, ta **đảo dấu value khi đổi người** — góc nhìn luôn là người hiện tại (encode_v3 đã chuẩn hoá theo người hiện tại), nhưng value phải được tính theo góc nhìn của *người sẽ đi tiếp*. Cách chuẩn: $V$ đầu ra là góc nhìn người hiện tại của state đó, và ta dùng quy tắc sign-flip giữa các bước.

In [36]:
def selfplay_game(net, hp, opponent=None):
    s = State(player=random.randrange(2))
    train_both = opponent is None
    agent_player = None if train_both else random.randrange(2)
    trajectory = []
    seen = {state_key(s)}
    outcome, reason = None, 'max_plies'
    while s.winner is None and s.ply < hp['max_plies']:
        train_actor = train_both or s.player == agent_player
        temp = hp['ep_temp'] if s.ply < hp['temp_floor_move'] else 0.5
        if not train_actor and opponent == 'heuristic':
            action = search_heuristic_action(s, seen=seen, hp=hp)
            canonical_action = canonical_actions([action], s.player)[0]
            mask = legal_mask(s, seen if hp.get('avoid_repetition', True) else None)
        else:
            actor = net if train_actor else opponent
            action, canonical_action, mask, _, _ = select_action(
                actor, s, temperature=temp, sample=True,
                noise=False, hp=hp, seen=seen)
        next_s = apply_unchecked(s, action)
        next_key = state_key(next_s)
        repeated = hp['repetition_draw'] and next_s.winner is None and next_key in seen
        terminal_next = (next_s.winner is not None or repeated or next_s.ply >= hp['max_plies'])
        reward = shaping_reward(s, next_s, hp, terminal=terminal_next)
        trajectory.append(A2CStep(encode_v3(s), mask, canonical_action,
                                  s.player, train_actor, reward, temp))
        s = next_s
        if s.winner is not None:
            outcome, reason = s.winner, 'win'
            break
        if repeated:
            reason = 'repetition'
            break
        seen.add(next_key)
    if s.winner is not None:
        outcome, reason = s.winner, 'win'
    if outcome is not None:
        trajectory[-1].reward += 1.0 if trajectory[-1].player == outcome else -1.0
    result = (0.0 if outcome is None else (1.0 if outcome == 0 else -1.0)) if train_both else (
        0.0 if outcome is None else (1.0 if outcome == agent_player else -1.0))
    metrics = {'result': result, 'end_reason': reason, 'length': len(trajectory),
               'repetition': reason == 'repetition',
               'walls_used': 20 - s.walls0 - s.walls1,
               'average_reward': float(np.mean([x.reward for x in trajectory]))}
    return trajectory, metrics

@torch.no_grad()
def compute_gae(net, game, hp):
    if not game:
        raise ValueError('Không thể tính GAE cho ván rỗng')
    obs = np.stack([step.observation for step in game]).astype(np.float32)
    old_mode = net.training
    net.eval()
    values = []
    for start in range(0, len(game), 128):
        x = torch.from_numpy(obs[start:start + 128]).to(device)
        _, v = net(x)
        if not torch.isfinite(v).all():
            raise FloatingPointError('Value prediction có NaN/Inf')
        values.extend(v.detach().cpu().numpy().reshape(-1).tolist())
    if old_mode:
        net.train()
    values = np.asarray(values, dtype=np.float32)
    rewards = np.asarray([step.reward for step in game], dtype=np.float32)
    # State sau mỗi nước thuộc lượt đối thủ; terminal/truncation đều có V_next=0.
    v_next = np.concatenate((-values[1:], np.zeros(1, dtype=np.float32)))
    delta = rewards + hp['gamma'] * v_next - values
    adv = np.zeros_like(delta)
    acc = 0.0
    for t in reversed(range(len(game))):
        acc = delta[t] - hp['gamma'] * hp['gae_lambda'] * acc
        adv[t] = acc
    ret = adv + values
    if not np.isfinite(adv).all() or not np.isfinite(ret).all():
        raise FloatingPointError('Advantage/return có NaN/Inf')
    return adv, ret

def build_dataset(buffer, hp):
    games = [game for game in buffer if game]
    rows = [step for game in games for step in game]
    if not rows:
        raise ValueError('Self-play tạo dataset rỗng')
    advs, returns = [], []
    for game in games:
        a, r = compute_gae(net, game, hp)
        advs.extend(a.tolist()); returns.extend(r.tolist())
    advs = np.asarray(advs, dtype=np.float32)
    actor_rows = np.asarray([step.train_actor for step in rows], dtype=np.bool_)
    if not actor_rows.any():
        raise ValueError('Dataset không có action của policy đang train')
    mean, std = advs[actor_rows].mean(), advs[actor_rows].std()
    advs[actor_rows] = ((advs[actor_rows] - mean) / std if std > 1e-8
                        else advs[actor_rows] - mean)
    advs[~actor_rows] = 0.0
    return TensorDataset(
        torch.from_numpy(np.stack([x.observation for x in rows]).astype(np.float32)),
        torch.from_numpy(np.stack([x.legal_mask for x in rows]).astype(np.bool_)),
        torch.tensor([x.action for x in rows], dtype=torch.long),
        torch.from_numpy(advs), torch.tensor(returns, dtype=torch.float32),
        torch.from_numpy(actor_rows),
        torch.tensor([x.temperature for x in rows], dtype=torch.float32))

## 5. Vòng lặp A2C

Loss tổng:

$$\mathcal{L} = -\underbrace{\mathbb{E}[\log\pi(a|s)\,\hat A]}_{\text{policy (actor)}} + \underbrace{c_v (G - V(s))^2}_{\text{value (critic)}} - \underbrace{c_e\, H[\pi]}_{\text{entropy bonus}}$$

Actor loss lấy `logp.gather()` tại action đã chọn và nhân với `advantage.detach()`; legal mask vẫn áp dụng cho cả policy lẫn entropy.

In [37]:
def a2c_loss(logits, value, masks, actions, advs, rets, actor_rows, temperatures, hp):
    if not torch.isfinite(logits).all() or not torch.isfinite(value).all():
        raise FloatingPointError('Policy logits/value có NaN/Inf')
    if not torch.isfinite(advs).all() or not torch.isfinite(rets).all():
        raise FloatingPointError('Advantage/return có NaN/Inf')
    masks = masks.bool()
    actions = actions.long().reshape(-1)
    actor_rows = actor_rows.bool().reshape(-1)
    temperatures = temperatures.float().reshape(-1)
    if not torch.isfinite(temperatures).all() or (temperatures <= 0).any():
        raise ValueError('Temperatures phải hữu hạn và > 0')
    if not masks.any(dim=1).all() or not masks.gather(1, actions[:, None]).all():
        raise ValueError('Trajectory chứa action không hợp lệ theo legal mask')
    adjusted_logits = action_prior_logits(logits / temperatures[:, None], masks, hp)
    logp = F.log_softmax(masked_policy(adjusted_logits, masks), dim=-1)
    chosen_logp = logp.gather(1, actions[:, None]).squeeze(1)
    policy_loss = logits.new_zeros(())
    if actor_rows.any():
        policy_loss = -(chosen_logp[actor_rows] * advs[actor_rows].detach()).mean()
        policy_loss = hp['policy_coef'] * policy_loss
    value_loss = F.mse_loss(value.reshape(-1), rets.detach())
    probs = logp.exp()
    entropy_by_row = -(probs * logp).sum(-1)
    entropy = entropy_by_row[actor_rows].mean() if actor_rows.any() else logits.new_zeros(())
    total = policy_loss + hp['value_coef'] * value_loss - hp['entropy_coef'] * entropy
    if not torch.isfinite(total):
        raise FloatingPointError('Total loss có NaN/Inf')
    return total, policy_loss, value_loss, entropy

def train(net, opt, ds, hp):
    if len(ds) == 0:
        raise ValueError('Không thể train với dataset rỗng')
    # Rollout uses eval mode; keep BatchNorm behaviour identical during the on-policy loss.
    net.eval()
    dl = DataLoader(ds, batch_size=hp['batch_size'], shuffle=True, drop_last=False)
    stats = {'policy_loss': 0.0, 'value_loss': 0.0, 'entropy': 0.0,
             'grad_norm': 0.0, 'batches': 0}
    for _epoch in range(hp['epochs_per_iter']):
        for obs, masks, actions, advs, rets, actor_rows, temperatures in dl:
            obs, masks = obs.to(device), masks.to(device)
            actions, advs = actions.to(device), advs.to(device)
            rets, actor_rows = rets.to(device), actor_rows.to(device)
            temperatures = temperatures.to(device)
            logits, value = net(obs)
            loss, pl, vl, en = a2c_loss(
                logits, value, masks, actions, advs, rets, actor_rows, temperatures, hp)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            grads = [p.grad for p in net.parameters() if p.grad is not None]
            if not grads or any(not torch.isfinite(g).all() for g in grads):
                raise FloatingPointError('Gradient có NaN/Inf hoặc bị thiếu')
            grad_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), hp['grad_clip'])
            if not torch.isfinite(grad_norm):
                raise FloatingPointError('Gradient norm có NaN/Inf')
            opt.step()
            stats['policy_loss'] += float(pl.detach())
            stats['value_loss'] += float(vl.detach())
            stats['entropy'] += float(en.detach())
            stats['grad_norm'] += float(grad_norm.detach())
            stats['batches'] += 1
    if stats['batches'] == 0:
        raise RuntimeError('DataLoader không tạo batch huấn luyện')
    for key in ('policy_loss', 'value_loss', 'entropy', 'grad_norm'):
        stats[key] /= stats['batches']
    return stats

## 6. Chạy huấn luyện

In [38]:
if HP['teacher_pretrain'] and start_iteration == 0 and teacher_pretrained_for is not net:
    print('Bắt đầu teacher pretraining trên mạng hiện tại...')
    behavior_clone(net, opt, HP)
    teacher_pretrained_for = net
elif HP['teacher_pretrain'] and start_iteration > 0:
    print('Bỏ qua teacher pretraining: đang tiếp tục từ checkpoint đã train.')
elif HP['teacher_pretrain']:
    print('Bỏ qua teacher pretraining: mạng này đã được pretrain trong kernel hiện tại.')
else:
    print('Teacher pretraining đang tắt.')

opponent_pool = OpponentPool(HP['opponent_pool_size'])
opponent_net = copy.deepcopy(net).to(device).eval()
opponent_snapshot_dir = RUN_DIR / 'checkpoints' / 'a2c_opponents'
opponent_snapshot_dir.mkdir(parents=True, exist_ok=True)
opponent_pool.load_directory(opponent_snapshot_dir, opponent_net)
training_csv = RUN_DIR / 'a2c_training.csv'
for it in range(start_iteration, HP['iterations']):
    print(f'=== iteration {it + 1}/{HP["iterations"]} ===')
    games, game_stats = [], []
    for _ in range(HP['games_per_iter']):
        opponent = None
        opponent_draw = random.random()
        if opponent_draw < HP['heuristic_opponent_fraction']:
            opponent = 'heuristic'
        elif (len(opponent_pool)
              and opponent_draw < HP['heuristic_opponent_fraction'] + HP['checkpoint_opponent_fraction']):
            _, old_state = opponent_pool.sample()
            opponent_net.load_state_dict(old_state)
            opponent = opponent_net
        game, metrics = selfplay_game(net, HP, opponent=opponent)
        games.append(game)
        game_stats.append(metrics)
    ds = build_dataset(games, HP)
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    losses = train(net, opt, ds, HP)
    losses['gpu_peak_memory_gb'] = (torch.cuda.max_memory_reserved(device) / 1024**3
                                    if device.type == 'cuda' else 0.0)
    recalibrate_batch_norm(net, ds.tensors[0], HP)
    count = len(game_stats)
    wins = sum(m['result'] > 0 for m in game_stats)
    draws = sum(m['result'] == 0 for m in game_stats)
    row = {
        'iteration': it + 1, 'games': count, 'samples': len(ds), **losses,
        'average_reward': float(np.mean([m['average_reward'] for m in game_stats])),
        'win_rate': wins / count, 'draw_rate': draws / count,
        'average_game_length': float(np.mean([m['length'] for m in game_stats])),
        'repetition_rate': float(np.mean([m['repetition'] for m in game_stats])),
        'average_walls': float(np.mean([m['walls_used'] for m in game_stats])),
        'learning_rate': float(opt.param_groups[0]['lr']),
    }
    history.append(row)
    print('  ' + ' | '.join(f'{key}={value:.4f}' if isinstance(value, float) else f'{key}={value}'
                           for key, value in row.items()))
    write_header = not training_csv.exists()
    with training_csv.open('a', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    checkpoint = {'model': net.state_dict(), 'optimizer': opt.state_dict(),
                  'hp': HP, 'iteration': it + 1, 'history': history}
    temporary = RUN_DIR / 'a2c_quoridor.pt.tmp'
    torch.save(checkpoint, temporary)
    os.replace(temporary, RUN_DIR / 'a2c_quoridor.pt')
    if (it + 1) % HP['opponent_snapshot_interval'] == 0:
        snapshot_path = opponent_snapshot_dir / f'a2c_iter_{it + 1:04d}.pt'
        opponent_pool.add(net, it + 1, path=str(snapshot_path))
print(f'Đã lưu checkpoint và CSV trong {RUN_DIR}')

Bỏ qua teacher pretraining: đang tiếp tục từ checkpoint đã train.
=== iteration 1025/1536 ===
BatchNorm recalibrated on 1136 states.
  iteration=1025 | games=64 | samples=1136 | policy_loss=-0.0573 | value_loss=0.4126 | entropy=0.1724 | grad_norm=6.1587 | batches=2 | gpu_peak_memory_gb=1.8398 | average_reward=0.0644 | win_rate=0.9062 | draw_rate=0.0000 | average_game_length=17.7500 | repetition_rate=0.0000 | average_walls=1.4531 | learning_rate=0.0001
=== iteration 1026/1536 ===
BatchNorm recalibrated on 1569 states.
  iteration=1026 | games=64 | samples=1569 | policy_loss=-0.2335 | value_loss=0.6255 | entropy=0.8324 | grad_norm=2.9002 | batches=2 | gpu_peak_memory_gb=1.8398 | average_reward=0.0535 | win_rate=0.7500 | draw_rate=0.0000 | average_game_length=24.5156 | repetition_rate=0.0000 | average_walls=4.2812 | learning_rate=0.0001
=== iteration 1027/1536 ===
BatchNorm recalibrated on 1182 states.
  iteration=1027 | games=64 | samples=1182 | policy_loss=-0.1865 | value_loss=0.3580 | 

KeyboardInterrupt: 

## 7. Evaluation đối kháng

Đánh giá greedy, không Dirichlet/temperature sampling: AI vs teacher heuristic (đường BFS + xét phản đòn một ply trên top action), Random, checkpoint cũ và AI. Mỗi matchup đổi bên đi trước/sau và mặc định 100 ván; kết quả được lưu CSV.

In [ ]:
@torch.no_grad()
def evaluate_suite(model, previous, hp, games=100, csv_path='a2c_evaluation.csv'):
    if games < 100:
        raise ValueError('Evaluation chính thức cần ít nhất 100 ván mỗi đối thủ')
    results = []
    for mode, opponent in (('heuristic', 'heuristic'), ('random', None),
                           ('previous', previous), ('ai', model)):
        per_game = []
        for g in range(games):
            candidate_player = g % 2  # cân bằng bên đi trước và đi sau
            s = State(player=candidate_player)
            seen = {state_key(s)}
            reason = 'max_plies'
            while s.winner is None and s.ply < hp['max_plies']:
                if s.player == candidate_player:
                    action = select_action(model, s, sample=False, noise=False,
                                           hp=hp, seen=seen)[0]
                elif mode == 'random':
                    random_seen = seen if hp.get('avoid_repetition', True) else None
                    action = random.choice(repetition_safe_actions(s, random_seen))
                elif mode == 'heuristic':
                    action = search_heuristic_action(s, seen=seen, hp=hp)
                else:
                    action = select_action(opponent, s, sample=False, noise=False,
                                           hp=hp, seen=seen)[0]
                s = apply_unchecked(s, action)
                if s.winner is not None:
                    reason = 'win'
                    break   ư
                key = state_key(s)
                if hp['repetition_draw'] and key in seen:
                    reason = 'repetition'
                    break
                seen.add(key)
            result = 1 if s.winner == candidate_player else (-1 if s.winner is not None else 0)
            per_game.append({'result': result, 'length': s.ply, 'repetition': reason == 'repetition',
                             'walls': 20 - s.walls0 - s.walls1})
        wins = sum(x['result'] > 0 for x in per_game)
        draws = sum(x['result'] == 0 for x in per_game)
        results.append({
            'opponent': mode, 'games': games, 'win_rate': wins / games,
            'draw_rate': draws / games, 'loss_rate': (games - wins - draws) / games,
            'average_game_length': float(np.mean([x['length'] for x in per_game])),
            'repetition_rate': float(np.mean([x['repetition'] for x in per_game])),
            'average_walls': float(np.mean([x['walls'] for x in per_game])),
        })
    path = Path(csv_path)
    write_header = not path.exists()
    with path.open('a', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(results[0]))
        if write_header:
            writer.writeheader()
        writer.writerows(results)
    return results

def plot_history(rows, path='a2c_training.png'):
    import matplotlib.pyplot as plt
    fields = [('policy_loss', 'Policy loss'), ('value_loss', 'Value loss'),
              ('entropy', 'Entropy'), ('win_rate', 'Win rate'),
              ('draw_rate', 'Draw rate'), ('average_game_length', 'Game length')]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (field, title) in zip(axes.flat, fields):
        ax.plot([row['iteration'] for row in rows], [row.get(field, np.nan) for row in rows])
        ax.set_title(title); ax.set_xlabel('Iteration'); ax.grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

previous_net = copy.deepcopy(net).to(device).eval()
previous_snapshots = [(iteration, state) for iteration, state in opponent_pool.snapshots
                      if iteration < history[-1]['iteration']]
if previous_snapshots:
    _, previous_state = max(previous_snapshots, key=lambda item: item[0])
    previous_net.load_state_dict(previous_state)
evaluation_results = evaluate_suite(
    net, previous_net, HP, games=HP['evaluation_games'],
    csv_path=RUN_DIR / 'a2c_evaluation.csv')
for result in evaluation_results:
    print(result)
plot_history(history, path=RUN_DIR / 'a2c_training.png')
print(f'Đã lưu evaluation CSV và training plot trong {RUN_DIR}')

{'opponent': 'heuristic', 'games': 100, 'win_rate': 0.0, 'draw_rate': 0.0, 'loss_rate': 1.0, 'average_game_length': 14.0, 'repetition_rate': 0.0, 'average_walls': 0.0}
{'opponent': 'random', 'games': 100, 'win_rate': 0.76, 'draw_rate': 0.21, 'loss_rate': 0.03, 'average_game_length': 30.83, 'repetition_rate': 0.21, 'average_walls': 9.0}
{'opponent': 'previous', 'games': 100, 'win_rate': 0.0, 'draw_rate': 0.0, 'loss_rate': 1.0, 'average_game_length': 14.0, 'repetition_rate': 0.0, 'average_walls': 0.0}
{'opponent': 'ai', 'games': 100, 'win_rate': 0.0, 'draw_rate': 0.0, 'loss_rate': 1.0, 'average_game_length': 14.0, 'repetition_rate': 0.0, 'average_walls': 0.0}
Đã lưu evaluation CSV và training plot trong results\a2c_20261107_115542


## 8. Giải thích các thông số của thuật toán RL Actor-Critic

### a) `gamma` — hệ số chiết khấu (discount factor), 0 < γ < 1
Đo **mức quan trọng của tương lai**. Reward cách $t$ bước được nhân thêm $\gamma^t$.
- `γ = 0` → agent chỉ quan tâm phần thưởng ngay lập tức (short-sighted).
- `γ = 1` → coi tương lai quan trọng ngang hiện tại, thường gây nhiễu.
- Trong Quoridor (game có chiều dài rõ ràng, chỉ thắng/thua ở cuối), `γ = 0.99` là chuẩn: một ván ~100-160 nước nên $0.99^{160} \approx 0.2$, tức phần thưởng thắng vẫn còn tín hiệu đáng kể ở cuối ván.

### b) `gae_lambda` — lambda của GAE, 0 ≤ λ ≤ 1
Điều chỉnh **độ dài của bootstrap** khi ước lượng advantage $\hat A_t$:
- `λ = 0` → thuần TD(0): chỉ dùng TD error 1 bước (`δ_t`), bias cao, variance thấp.
- `λ = 1` → thuần Monte-Carlo: dùng toàn bộ reward thực tế đến cuối ván, bias thấp, variance cao.
- `λ = 0.95` là điểm cân bằng tốt cho board game: đủ ổn định nhưng không quá nhiễu.

### c) `entropy_coef` — hệ số entropy bonus ($c_e$)
Khuyến khích policy **không quá tự tin sớm**. Nếu policy quá sớm collapse về một action, agent ngừng khám phá và dính local optimum. Loss có thêm thuật ngữ $-c_e \cdot H[\pi]$; giá trị lớn hơn → khám phá nhiều hơn, nhưng nếu quá lớn policy sẽ học chậm / gần như ngẫu nhiên. Rất hay decay dần `c_e` theo thời gian (khám phá lúc đầu, exploiting sau).

### d) `value_coef` — hệ số value loss ($c_v$)
Trọng số của loss của **critic** trong tổng loss. Critic là "người chấm điểm" dạy cho actor; nếu critic học quá nhanh và quá mạnh sẽ dẫn policy đi sai hướng. `0.5` là chuẩn của A2C / PPO.

### e) `policy_coef` — hệ số policy loss
Trọng số cho phần loss của **actor**. A2C gốc đặt bằng 1, còn PPO đặt là $1/\epsilon$ sau khi chuẩn hoá.

### f) Value loss
Notebook dùng MSE giữa value và return target; không bật value clipping. NaN/Inf được kiểm tra và gradient được clip theo `grad_clip`.

### g) `lr` (learning rate)
Bước đi trên gradient. Quá lớn → loss phát xung, quên nhanh; quá nhỏ → học chậm. Ở đây dùng `AdamW` + `grad_clip` để ổn định; nên decay theo iteration.

### h) `batch_size` và `epochs_per_iter`
- `batch_size`: số rollout mẫu mỗi lần cập nhật A2C; batch lớn giúp hạn chế policy drift giữa các minibatch. `teacher_batch_size` được dùng riêng cho behavior cloning.
- `epochs_per_iter`: số lần học lại trên cùng dữ liệu. Vì buffer là on-policy, notebook dùng một epoch để giảm lệch khỏi policy đã tạo rollout.

### i) `grad_clip` — max grad norm
Chặn gradient quá lớn (exploding gradient) do value target ngoài range hoặc bootstrap lỗi.

### j) `weight_decay` (AdamW)
Regularization L2, chống overfit trên buffer nhỏ.

### k) `ep_temp` (temperature)
Điều chỉnh **mức ngẫu nhiên lúc thu dữ liệu**. $p(a) \propto \pi(a)^{1/T}$:
- `T cao` → gần uniform, khám phá nhiều.
- `T thấp` → gần greedy (argmax), khai thác nhiều.
- Notebook này dùng `T = 1` ở các nước đầu, giảm còn `0.5` sau đó; khi đánh giá ta dùng `T → 0` (greedy).

### l) Temperature và Dirichlet noise
Self-play lưu temperature của từng nước và dùng lại đúng temperature đó trong loss. Dirichlet noise được tắt trong A2C rollout để action được lấy mẫu từ chính policy đang tối ưu; `select_action` vẫn hỗ trợ noise cho thử nghiệm riêng.

`pawn_action_prior` chia policy thành nhóm đi quân và đặt tường, tránh để số lượng lựa chọn đặt tường áp đảo ngay từ đầu.

### m) `max_plies`
Giới hạn số ply mỗi ván; chạm giới hạn được xem là hoà terminal trong rollout. Đây khác với repetition detection.

### n) `channels` và `blocks` của mạng
- `channels`: độ rộng mỗi conv layer — quyết định capacity của model.
- `blocks`: số residual block — quyết định độ sâu (khả năng học quan hệ dài giữa các ô).
Model lớn hơn cần nhiều dữ liệu hơn, nhưng học được chiến lược sâu hơn.

### o) `value_blend_q` / tham số AlphaZero của project
Project gốc (`az_train.py`) còn có `value_blend_q` để trộn giữa terminal reward `z` và Q ước lượng từ MCTS; phần A2C của notebook này không dùng MCTS nên không tham số đó.

### Tóm tắt vai trò từng thành phần
| Thành phần | Vai trò | Tinh chỉnh thế nào |
|---|---|---|
| `gamma` | chiết khấu tương lai | game ngắn → cao, dài → thấp |
| `gae_lambda` | bias-variance advantage | càng cao càng ít bias, nhiều variance |
| `entropy_coef` | chống premature convergence | decay dần theo thời gian |
| `value_coef` | cân bằng actor-critic | critic quá mạnh → giảm |
| `reward_shaping` | hướng dẫn tiến triển BFS | bật/tắt; terminal reward giữ nguyên |
| `repetition_draw` | kết thúc cycle | xem repetition là hoà trong rollout |
| `avoid_repetition` | loại nước lặp nếu còn nước khác | không đổi legal actions của engine |
| `wall_value_weight` | giữ giá trị tường còn lại trong shaping | tường hữu ích vẫn có thể đáng đặt |
| `pawn_action_prior` | cân bằng nhóm đi quân/tường | prior hiện tại 0.9 / 0.1 |
| `teacher_pretrain` | behavior cloning từ teacher trước A2C | bật cho lần train mới |
| `teacher_games` / `teacher_epochs` | dữ liệu và số lượt bắt chước teacher | tăng nếu accuracy teacher thấp |
| `teacher_search_width` | số action đầu được xét phản đòn một ply | rộng hơn tốn thời gian hơn |
| `lr` | tốc độ học | decay theo iteration |
| `batch_size` | noise gradient | lớn hơn nếu buffer lớn |
| `epochs_per_iter` | on-policy constraint | 1–3 |
| `grad_clip` | chống exploding gradient | 0.5–1.0 |
| `ep_temp` | khám phá lúc thu data | cao lúc đầu, thấp sau |
| `dirichlet_alpha` | đảm bảo khám phá đều | 0.3 cho 209 actions |

### Về actor & critic
- **Actor** = policy head: cho xác suất chọn từng action ở mỗi thế cờ.
- **Critic** = value head: cho ước lượng giá trị thế cờ $V(s)$.
- **Advantage** $A(s,a) = Q(s,a) - V(s)$: cho biết action $a$ tốt hơn bình thường bao nhiêu. Dùng advantage thay vì return thô giúp giảm variance và học nhanh hơn — đó chính là chữ **A** trong **A2C**.